### Read the CSV with all packets, add Class column (if does not exist) and export it to be used by the Replace_packet_fields

In [3]:
import pandas as pd
import sys
import os

file_path_all = "data/" + \
           "data_and_models/PCAP(DL-UL)/DS_Kombat_DL_UL.csv"

# if Kombat in file_path_all, then game = "Kombat"
if "Kombat" in file_path_all:
    game = "Kombat"
elif "Forza" in file_path_all:
    game = "Forza"
else:
    game = "Other"

print("Game: ", game)

# If other, quit
if game == "Other":
    print("Game not supported")
    sys.exit()


# Load your CSV
first_packets = 50
print("Looking at the first", first_packets, "packets")
df_all = pd.read_csv(file_path_all)
df_all = df_all[['IPI', 'Source', 'Destination', 'Protocol']]
df_first_packets = df_all.head(first_packets)

# Find most common IP pairs
pair_counts = df_first_packets.groupby(['Source', 'Destination']).size().reset_index(name='count')
top_pair = pair_counts.sort_values('count', ascending=False).iloc[0]
ip1, ip2 = top_pair['Source'], top_pair['Destination']

# Only consider UDP/RTP for directionality
udp_rtp = df_first_packets[df_first_packets['Protocol'].isin(['UDP', 'RTP'])]

# Count packets in each direction
count_ip1_to_ip2 = ((udp_rtp['Source'] == ip1) & (udp_rtp['Destination'] == ip2)).sum()
count_ip2_to_ip1 = ((udp_rtp['Source'] == ip2) & (udp_rtp['Destination'] == ip1)).sum()

if count_ip1_to_ip2 >= count_ip2_to_ip1:
    cg_server_ip, cg_client_ip = ip1, ip2
else:
    cg_server_ip, cg_client_ip = ip2, ip1

print(f"Detected server IP: {cg_server_ip}, client IP: {cg_client_ip}")

def get_direction(row):
    if row['Protocol'] in ['UDP', 'RTP']:
        if row['Source'] == cg_server_ip and row['Destination'] == cg_client_ip:
            return 'Downlink'
        elif row['Source'] == cg_client_ip and row['Destination'] == cg_server_ip:
            return 'Uplink'
    return 'Uplink'

df_all['Direction'] = df_all.apply(get_direction, axis=1)

# Show a sample
print(df_all[['IPI', 'Source', 'Destination', 'Protocol', 'Direction']].head(first_packets))

# keep only IPI and Direction columns
df_all = df_all[['IPI', 'Direction']]

# Save to CSV if not exists
if not os.path.exists("DS_" + game + "_all_packets.csv"):
    df_all.to_csv("DS_" + game + "_all_packets.csv", index=False)

Game:  Kombat
Looking at the first 50 packets
Detected server IP: 13.104.107.204, client IP: 200.18.102.42
             IPI          Source     Destination  Protocol Direction
0   0.000000e+00   200.18.102.42  13.104.107.204       UDP    Uplink
1   4.318293e-03   200.18.102.42  13.104.107.204  DTLSv1.2    Uplink
2   7.157740e-04  13.104.107.204   200.18.102.42       UDP  Downlink
3   6.900000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
4   2.100000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
5   1.300000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
6   1.400000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
7   1.300000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
8   1.300000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
9   1.500000e-08  13.104.107.204   200.18.102.42       UDP  Downlink
10  2.586500e-05  13.104.107.204   200.18.102.42       UDP  Downlink
11  1.700000e-08  13.104.107.204   200.18.102.42       UDP  Downl